In [0]:
from pyspark.sql import functions as F

SOURCE_BRONZE = "iotmlhealthcatalog.bronze.iot_stream_raw"
SILVER_TABLE = "iotmlhealthcatalog.silver.vitaldbstream"
STATE_TABLE = "iotmlhealthcatalog.bronze.patientslastsigns"
CHECKPOINT_PATH = "/Volumes/iotmlhealthcatalog/default/iotbatch/_checkpoints/silver_state_sync"

def update_state_and_silver(batch_df, batch_id):

    if batch_df.limit(1).count() == 0:
        return

    # =========================
    # 1. Préparation
    # =========================
    updates_df = batch_df.select(
        F.col("timestamp").cast("timestamp").alias("timestamp"),
        F.col("caseid").cast("integer").alias("caseid"),
        F.col("SNUADC_ART_SBP").alias("sbp"),
        F.col("Solar8000_HR").alias("hr"),
        F.col("Solar8000_PLETH_SPO2").alias("spo2"),
        F.col("Solar8000_BT").alias("temp"),
        F.col("ingestion_timestamp").alias("ingestion_timestamp")
    )

    # =========================
    # 2. Dernier état PAR PATIENT
    # =========================
    latest_updates = updates_df.groupBy("caseid").agg(
        F.max("timestamp").alias("last_update"),
        F.last("sbp", ignorenulls=True).alias("last_sbp"),
        F.last("hr", ignorenulls=True).alias("last_hr"),
        F.last("spo2", ignorenulls=True).alias("last_spo2"),
        F.last("temp", ignorenulls=True).alias("last_temp")
    )

    # =========================
    # 3. TABLE TEMP
    # =========================
    latest_updates.createOrReplaceTempView("updates_tmp")

    # =========================
    # 4. INIT TABLE SI ABSENTE
    # =========================
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {STATE_TABLE}
        USING DELTA
        AS SELECT * FROM updates_tmp WHERE 1=0
    """)

    # =========================
    # 5. MERGE SQL (SAFE)
    # =========================
    spark.sql(f"""
        MERGE INTO {STATE_TABLE} AS target
        USING updates_tmp AS source
        ON target.caseid = source.caseid

        WHEN MATCHED THEN UPDATE SET
            target.last_sbp = coalesce(source.last_sbp, target.last_sbp),
            target.last_hr = coalesce(source.last_hr, target.last_hr),
            target.last_spo2 = coalesce(source.last_spo2, target.last_spo2),
            target.last_temp = coalesce(source.last_temp, target.last_temp),
            target.last_update = source.last_update

        WHEN NOT MATCHED THEN INSERT *
    """)

    # =========================
    # 6. SILVER (IMPUTATION)
    # =========================
    current_state = spark.table(STATE_TABLE).alias("state")
    updates_df = updates_df.alias("upd")

    silver_final = updates_df.join(current_state, "caseid", "left") \
        .select(
            F.col("upd.caseid"),
            F.col("upd.timestamp"),
            F.coalesce(F.col("upd.sbp"), F.col("state.last_sbp")).alias("sbp"),
            F.coalesce(F.col("upd.hr"), F.col("state.last_hr")).alias("hr"),
            F.coalesce(F.col("upd.spo2"), F.col("state.last_spo2")).alias("spo2"),
            F.coalesce(F.col("upd.temp"), F.col("state.last_temp")).alias("temp")
        )

    # =========================
    # 7. WRITE SILVER
    # =========================
    silver_final.write.format("delta").mode("append").saveAsTable(SILVER_TABLE)


# =========================
# STREAM
# =========================
query = (spark.readStream.table(SOURCE_BRONZE)
    .writeStream
    .foreachBatch(update_state_and_silver)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()